In [152]:
!pip install geemap earthengine-api ipyleaflet ipywidgets -q


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [153]:
import ee
import geemap
import ipywidgets as widgets

ee.Authenticate()
ee.Initialize()

# bounds = [-4.04, 51.54, -3.32, 51.75] wales test
# bounds = [-4.95, 57.18, -4.86, 57.27] Glenn Africk - not valid due to 2016 clouds
bounds = [109.35, 36.45, 109.44, 36.54]

coord1 = (bounds[1] + bounds[3]) / 2 
coord2 = (bounds[0] + bounds[2]) / 2

Map = geemap.Map(center=[coord1, coord2], zoom=10)

In [154]:
TILE_PIXELS = 128
SCALE = 10
TILE_METERS = TILE_PIXELS * SCALE

def build_aoi(bounds):
    raw = ee.Geometry.Rectangle(bounds)
    coords = ee.List(raw.coordinates().get(0))

    raw_min_lon = ee.Number(ee.List(coords.get(0)).get(0))
    raw_min_lat = ee.Number(ee.List(coords.get(0)).get(1))
    raw_max_lon = ee.Number(ee.List(coords.get(2)).get(0))
    raw_max_lat = ee.Number(ee.List(coords.get(2)).get(1))

    tile_deg_lat = ee.Number(TILE_METERS).divide(111320)
    center_lat = raw_min_lat.add(raw_max_lat).divide(2)
    lat_cos = center_lat.multiply(3.141592653589793 / 180).cos()
    tile_deg_lon = ee.Number(TILE_METERS).divide(111320).divide(lat_cos)

    min_lon = raw_min_lon.divide(tile_deg_lon).floor().multiply(tile_deg_lon)
    min_lat = raw_min_lat.divide(tile_deg_lat).floor().multiply(tile_deg_lat)
    max_lon = raw_max_lon.divide(tile_deg_lon).ceil().multiply(tile_deg_lon)
    max_lat = raw_max_lat.divide(tile_deg_lat).ceil().multiply(tile_deg_lat)

    aoi = ee.Geometry.Rectangle([min_lon, min_lat, max_lon, max_lat])

    return aoi, min_lon, min_lat, max_lon, max_lat, tile_deg_lon, tile_deg_lat


aoi, min_lon, min_lat, max_lon, max_lat, tile_deg_lon, tile_deg_lat = build_aoi(bounds)

In [155]:
def build_gain_layer(aoi):
    worldcover = ee.Image("ESA/WorldCover/v100/2020").clip(aoi)
    is_forest = worldcover.eq(10)

    m15 = ee.Image("projects/glad/GLCLU2020/v2/LCLUC_2015").clip(aoi)
    m20 = ee.Image("projects/glad/GLCLU2020/v2/LCLUC_2020").clip(aoi)

    tree_classes = ee.List.sequence(25, 96).cat(ee.List.sequence(125, 196))
    ones = ee.List.repeat(1, tree_classes.length())

    tree2015 = m15.remap(tree_classes, ones, 0)
    tree2020 = m20.remap(tree_classes, ones, 0)

    forest_gain = tree2020.And(tree2015.Not())
    clean_gain = forest_gain.updateMask(forest_gain).focal_max(1).focal_min(1)

    gain_validated = clean_gain.And(is_forest)
    gain_binary = gain_validated.unmask(0).rename("gain")

    return gain_validated, gain_binary, is_forest


gain_validated, gain_binary, is_forest = build_gain_layer(aoi)

In [156]:
import math

def build_grid(min_lon, min_lat, max_lon, max_lat, dx, dy):

    min_lon = ee.Number(min_lon)
    min_lat = ee.Number(min_lat)
    max_lon = ee.Number(max_lon)
    max_lat = ee.Number(max_lat)
    dx = ee.Number(dx)
    dy = ee.Number(dy)

    n_lon = max_lon.subtract(min_lon).divide(dx).floor()
    n_lat = max_lat.subtract(min_lat).divide(dy).floor()

    lon_vals = ee.List.sequence(0, n_lon.subtract(1)).map(
        lambda i: min_lon.add(ee.Number(i).multiply(dx))
    )

    lat_vals = ee.List.sequence(0, n_lat.subtract(1)).map(
        lambda j: min_lat.add(ee.Number(j).multiply(dy))
    )

    def make_row(lon):
        lon = ee.Number(lon)

        def make_cell(lat):
            lat = ee.Number(lat)

            geom = ee.Geometry.Rectangle(
                [lon, lat, lon.add(dx), lat.add(dy)],
                geodesic=False,
            )

            return ee.Feature(geom, {
                "tile_id": ee.String("tile_")
                    .cat(lon.format())
                    .cat("_")
                    .cat(lat.format())
            })

        return lat_vals.map(make_cell)

    return ee.FeatureCollection(lon_vals.map(make_row).flatten())

grid = build_grid(min_lon, min_lat, max_lon, max_lat, tile_deg_lon, tile_deg_lat)

In [157]:
def build_valid_tiles(gain_binary, full_valid, grid):

    tile_area_pixels = (
        ee.Number(TILE_METERS)
        .divide(SCALE)
        .pow(2)
    )

    gain_count = gain_binary.reduceRegions(
        collection=grid,
        reducer=ee.Reducer.sum(),
        scale=SCALE,
        tileScale=4,
    )

    valid_tiles = (
        full_valid
        .unmask(0)
        .reduceRegions(
            collection=gain_count,
            reducer=ee.Reducer.min(),
            scale=SCALE,
            tileScale=4,
        )
        .filter(
            ee.Filter.eq("min", 1)
        )
    )

    return valid_tiles, tile_area_pixels

In [158]:
def add_indices(img):
    ndvi = img.normalizedDifference(["B8", "B4"]).rename("NDVI")
    evi = img.expression(
        "2.5 * ((NIR - RED) / (NIR + 6.0 * RED - 7.5 * BLUE + 1.0))",
        {"NIR": img.select("B8"), "RED": img.select("B4"), "BLUE": img.select("B2")},
    ).rename("EVI")

    return img.select(["B2","B3","B4","B5","B6","B7","B8"]).addBands([ndvi, evi])


def s2_composite(geom, year):
    start = "2015-01-01" if year == 2016 else f"{year}-01-01"
    end = "2016-12-31" if year == 2016 else f"{year}-12-31"
    ic = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterDate(start, end)
        .filterBounds(geom)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 30))
        .map(add_indices)
    )
    reduced = ic.reduce(ee.Reducer.percentile([25]))

    fallback = ee.Image.constant(0).rename("B2_p25") \
        .addBands(ee.Image.constant(0).rename("B3_p25")) \
        .addBands(ee.Image.constant(0).rename("B4_p25")) \
        .addBands(ee.Image.constant(0).rename("B5_p25")) \
        .addBands(ee.Image.constant(0).rename("B6_p25")) \
        .addBands(ee.Image.constant(0).rename("B7_p25")) \
        .addBands(ee.Image.constant(0).rename("B8_p25")) \
        .addBands(ee.Image.constant(0).rename("NDVI_p25")) \
        .addBands(ee.Image.constant(0).rename("EVI_p25")) \
        .updateMask(ee.Image.constant(0))  # all masked = no valid pixels
    reduced = ee.Algorithms.If(ic.size().eq(0), fallback, reduced)
    reduced = ee.Image(reduced)
    return reduced.select(
        ["B2_p25","B3_p25","B4_p25","B5_p25","B6_p25","B7_p25","B8_p25","NDVI_p25","EVI_p25"],
        ["B2","B3","B4","B5","B6","B7","B8","NDVI","EVI"],
    )


def s2_peak_composite(geom, year):
    centroid = ee.Geometry(geom).centroid(maxError=1)
    lat = ee.Number(centroid.coordinates().get(1))
    north = lat.gt(0)

    if year == 2016:
        start = ee.String(ee.Algorithms.If(north, "2015-05-01", "2015-11-01"))
        end   = ee.String(ee.Algorithms.If(north, "2016-09-30", "2017-03-31"))
    else:
        start = ee.String(ee.Algorithms.If(north, f"{year}-05-01", f"{year}-11-01"))
        end   = ee.String(ee.Algorithms.If(north, f"{year}-09-30", f"{year+1}-03-31"))

    return (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterDate(start, end)
        .filterBounds(geom)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 50))
        .map(add_indices)
        .select(["NDVI", "EVI"])
        .median()
    )


def s1_composite(geom, year):
    med = (
        ee.ImageCollection("COPERNICUS/S1_GRD")
        .filterDate(f"{year}-01-01", f"{year}-12-31")
        .filterBounds(geom)
        .filter(ee.Filter.eq("instrumentMode", "IW"))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
        .select(["VV", "VH"])
        .median()
    )
    return med.addBands(med.select("VV").divide(med.select("VH")).rename("VVVH"))


def dw_composite(geom, year):
    return (
        ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1")
        .filterDate(f"{year}-01-01", f"{year}-12-31")
        .filterBounds(geom)
        .select(["trees", "crops", "built"])
        .median()
    )


s2_2016 = s2_composite(aoi, 2016)
s2_2020 = s2_composite(aoi, 2020)
s2_2025 = s2_composite(aoi, 2025)

In [159]:
def score_tile_viability(tile, gain_validated, gain_height):

    geom = tile.geometry()
    gain_mask = gain_validated.selfMask()

    s2_t0 = s2_peak_composite(geom, 2016)
    s2_t1 = s2_peak_composite(geom, 2020)

    ndvi_diff = (
        s2_t1.select("NDVI")
        .subtract(s2_t0.select("NDVI"))
        .updateMask(gain_mask)
        .rename("NDVI_diff")
    )

    stats = ndvi_diff.reduceRegion(
        reducer=ee.Reducer.median(),
        geometry=geom,
        scale=SCALE,
        maxPixels=1e13,
    )

    ndvi_delta = ee.Number(
        ee.Algorithms.If(
            stats.get("NDVI_diff"),
            stats.get("NDVI_diff"),
            0,
        )
    )

    canopy_stats = (
        gain_height
        .updateMask(gain_mask)
        .reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=geom,
            scale=SCALE,
            maxPixels=1e13,
        )
    )

    canopy_mean = ee.Number(
        ee.Algorithms.If(
            canopy_stats.get("canopy_gain_height"),
            canopy_stats.get("canopy_gain_height"),
            0,
        )
    )

    return tile.set({
        "ndvi_delta": ndvi_delta,
        "gain_canopy_mean": canopy_mean,
    })

In [160]:
def s2_availability(geom, year):
    start = "2015-01-01" if year == 2016 else f"{year}-01-01"
    end = "2016-12-31" if year == 2016 else f"{year}-12-31"

    bands = ["B2","B3","B4","B5","B6","B7","B8"]

    ic = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterDate(start, end)
        .filterBounds(geom)
        .select(bands)
    )

    return ic.map(lambda img: img.mask().reduce(ee.Reducer.min())).reduce(ee.Reducer.max())


full_valid = (
    s2_availability(aoi, 2016)
    .And(s2_availability(aoi, 2020))
    .And(s2_availability(aoi, 2025))
    .selfMask()
)

In [161]:
fabdem = (
    ee.ImageCollection("projects/sat-io/open-datasets/FABDEM")
    .filterBounds(aoi)
    .mosaic()
    .clip(aoi)
)

slope = ee.Terrain.slope(fabdem)

canopy_raw = ee.Image("users/nlang/ETH_GlobalCanopyHeight_2020_10m_v1")
canopy_sd_raw = ee.Image("users/nlang/ETH_GlobalCanopyHeightSD_2020_10m_v1")

canopy = canopy_raw.clip(aoi).rename("canopy_height").updateMask(canopy_raw.gte(0))
canopy_sd = (
        canopy_sd_raw.clip(aoi).rename("canopy_height_sd").updateMask(canopy_raw.gte(0))
    )

gain_height = canopy.updateMask(gain_validated).rename("canopy_gain_height")
gain_height_sd = canopy_sd.updateMask(gain_validated).rename(
    "canopy_gain_height_sd"
)

jrc = ee.Image("JRC/GFC2020_subtypes/V1").clip(aoi).rename("jrc_forest_type")

nat_forest = (
    ee.ImageCollection(
        "projects/nature-trace/assets/forest_typology/natural_forest_2020_v1_0_collection"
    )
    .mosaic()
    .select("B0")
    .divide(250)
    .clip(aoi)
    .unmask(0)
    .rename("natural_forest_prob")
)

metrics_img = ee.Image.cat([
    gain_binary.rename("gain"),
    gain_height.rename("canopy"),
    jrc.rename("jrc"),
    nat_forest.rename("nat")
])

tile_metrics = metrics_img.reduceRegions(
    collection=grid,
    reducer=ee.Reducer.mean(),
    scale=SCALE,
    tileScale=4
)

jrc_mode = jrc.reduceRegions(
    collection=grid,
    reducer=ee.Reducer.mode(),
    scale=SCALE,
    tileScale=4
)

def merge(f):
    tid = f.get("system:index")
    jm = jrc_mode.filter(ee.Filter.eq("system:index", tid)).first()
    return f.set({"jrc_mode": jm.get("mode")})


tile_metrics = tile_metrics.map(merge)

GAIN_PCT_MIN  = 0.01
NDVI_DELTA_MIN = 0.0
GAIN_CANOPY_MIN = 3.0

gain_tiles = tile_metrics.filter(ee.Filter.gte("gain", GAIN_PCT_MIN))

scored_tiles = gain_tiles.map(
    lambda t: score_tile_viability(t, gain_validated, gain_height)
)

filtered_tiles = scored_tiles.filter(
    ee.Filter.And(
        ee.Filter.gt("ndvi_delta", NDVI_DELTA_MIN),
        ee.Filter.gte("gain_canopy_mean", GAIN_CANOPY_MIN),
    )
)

tiles_with_stats = filtered_tiles.map(lambda t: t.set({
    "gain_mean": gain_binary.reduceRegion(
        ee.Reducer.mean(), t.geometry(), SCALE, maxPixels=1e13
    ).get("gain"),

    "canopy_mean": gain_height.reduceRegion(
        ee.Reducer.mean(), t.geometry(), SCALE, maxPixels=1e13
    ).get("canopy_gain_height"),

     "canopy_sd_mean": gain_height_sd.reduceRegion(
        ee.Reducer.mean(), t.geometry(), SCALE, maxPixels=1e13
    ).get("canopy_gain_height_sd"),

    "jrc_mode": jrc.reduceRegion(
        ee.Reducer.mode(), t.geometry(), SCALE, maxPixels=1e13
    ).get("jrc_forest_type"),

    "nat_mean": nat_forest.reduceRegion(
        ee.Reducer.mean(), t.geometry(), SCALE, maxPixels=1e13
    ).get("natural_forest_prob"),
}))

In [162]:
print(
    scored_tiles.select([
        "tile_id",
        "ndvi_delta",
        "gain_canopy_mean",
    ]).getInfo()
)

{'type': 'FeatureCollection', 'columns': {}, 'features': [{'type': 'Feature', 'geometry': {'geodesic': False, 'type': 'Polygon', 'coordinates': [[[109.34715741069513, 36.449874236435505], [109.36146050518967, 36.449874236435505], [109.36146050518967, 36.46137261947539], [109.34715741069513, 36.46137261947539], [109.34715741069513, 36.449874236435505]]]}, 'id': '0', 'properties': {'gain_canopy_mean': 14.253658536585366, 'ndvi_delta': 0.6223858185112476, 'tile_id': 'tile_109.34715741069513_36.449874236435505'}}, {'type': 'Feature', 'geometry': {'geodesic': False, 'type': 'Polygon', 'coordinates': [[[109.43297597766232, 36.51886453467482], [109.44727907215686, 36.51886453467482], [109.44727907215686, 36.530362917714704], [109.43297597766232, 36.530362917714704], [109.43297597766232, 36.51886453467482]]]}, 'id': '48', 'properties': {'gain_canopy_mean': 12.777095988406154, 'ndvi_delta': 0.6294393486836377, 'tile_id': 'tile_109.43297597766232_36.51886453467482'}}]}


In [166]:
# Sentinel RGB
Map.addLayer(s2_2016.clip(aoi).select(["B4","B3","B2"]), {"min":200,"max":3000,"gamma":1.3}, "S2 2016")
Map.addLayer(s2_2020.clip(aoi).select(["B4","B3","B2"]), {"min":200,"max":3000,"gamma":1.3}, "S2 2020")
Map.addLayer(s2_2025.clip(aoi).select(["B4","B3","B2"]), {"min":200,"max":3000,"gamma":1.3}, "S2 2025")

# Gain
Map.addLayer(gain_validated.selfMask(), {"palette": ["00E676"]}, "Gain")

# Forest
Map.addLayer(is_forest.selfMask(), {"palette": ["006400"]}, "Forest 2020")

# Canopy
Map.addLayer(gain_height, {"min":0,"max":40,"palette":['f7fcf5','c7e9c0','74c476','238b45','00441b']}, "Canopy Gain Height")

# JRC
Map.addLayer(jrc, {"min":1,"max":20,"palette":['78c679','006837','cc6600']}, "JRC Forest Type")

# Natural forest
Map.addLayer(nat_forest, {"min":0,"max":1,"palette":['white','green']}, "Natural Forest Prob")

# Tiles
Map.addLayer(
    tiles_with_stats.style(color="00FF00", fillColor="00000000", width=2),
    {},
    "Filtered Tiles"
)

s2_2016_peak = s2_peak_composite(aoi, 2016)
s2_2020_peak = s2_peak_composite(aoi, 2020)

ndvi_diff = (
    s2_2020_peak.select("NDVI")
    .subtract(s2_2016_peak.select("NDVI"))
    .clip(aoi)
    .rename("NDVI_diff")
)

Map.addLayer(
    ndvi_diff,
    {"min": -1, "max": 1, "palette": ["red", "white", "green"]},
    "NDVI 2020 - 2016 (peak season)"
)

Map.addLayerControl()

output = widgets.Output(
    layout=widgets.Layout(
        width="280px",
        max_height="220px",
        overflow_y="auto",
        border="1px solid #ccc",
        padding="8px",
        background_color="white",
    )
)

with output:
    print("Click a tile to see statistics.")


JRC_LABELS = {
    1: "Naturally regenerating",
    10: "Primary forest",
    20: "Planted/Plantation",
}

def handle_click(**kwargs):
    if kwargs.get("type") != "click":
        return
    coords = kwargs["coordinates"]
    point = ee.Geometry.Point([coords[1], coords[0]])
    info = tiles_with_stats.filterBounds(point).first().getInfo()
    with output:
        output.clear_output()
        if not info:
            print("No filtered tile here.")
            return
        p = info["properties"]

        def fmt(v, d=2):
            return f"{float(v):.{d}f}" if v is not None else "N/A"

        jrc_raw = p.get("jrc_mode")
        jrc_label = JRC_LABELS.get(int(jrc_raw), "Unknown") if jrc_raw is not None else "N/A"

        print(f"Tile ID : {p.get('tile_id', 'N/A')}")
        print(f"Gain area (%) : {fmt(p.get('gain_mean') * 100)}%")
        print(f"Canopy (m) : {fmt(p.get('canopy_mean'))}")
        print(f"Natural forest (%) : {fmt(p.get('nat_mean'))}")
        print(f"JRC type : {jrc_label}")


Map.on_interaction(handle_click)
Map.add_widget(output, position="bottomright")

In [164]:
Map

Map(center=[36.495000000000005, 109.395], controls=(WidgetControl(options=['position', 'transparent_bg'], posi…